# CStats - Python Lab 10: Auto-Encoders 

**Libraries:** `numpy`, `matplotlib`, `torch`, `scikit-learn`.

If you are running locally and a package is missing:
```bash
pip install numpy matplotlib torch scikit-learn
```



---
## Setup

Run the next cell first. It imports libraries, downloads Fashion-MNIST, defines plotting helpers, and defines small PyTorch model-building utilities.


In [ ]:
import gzip
import struct
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report

SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

FASHION_LABELS = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Keep the lab fast. Increase these values for cleaner results.
TRAIN_SUBSET = 12000
TEST_SUBSET = 2500
BATCH_SIZE = 256

# ---------------------------------------------------------------------
# Data loading: pure PyTorch/Numpy, no TensorFlow and no torchvision needed.
# This avoids version mismatches in some notebook environments.
# ---------------------------------------------------------------------
FASHION_MNIST_BASE_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/"
FASHION_MNIST_FILES = {
    "train_images": "train-images-idx3-ubyte.gz",
    "train_labels": "train-labels-idx1-ubyte.gz",
    "test_images": "t10k-images-idx3-ubyte.gz",
    "test_labels": "t10k-labels-idx1-ubyte.gz",
}

def _download_if_needed(filename, root="./data/fashion-mnist"):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    path = root / filename
    if not path.exists():
        url = FASHION_MNIST_BASE_URL + filename
        print(f"Downloading {url}")
        urllib.request.urlretrieve(url, path)
    return path

def _read_idx_images(path):
    with gzip.open(path, "rb") as f:
        magic, n_images, rows, cols = struct.unpack(">IIII", f.read(16))
        if magic != 2051:
            raise ValueError(f"Unexpected magic number {magic} in image file {path}")
        data = np.frombuffer(f.read(), dtype=np.uint8).reshape(n_images, rows, cols)
    return torch.tensor(data, dtype=torch.float32).unsqueeze(1) / 255.0

def _read_idx_labels(path):
    with gzip.open(path, "rb") as f:
        magic, n_labels = struct.unpack(">II", f.read(8))
        if magic != 2049:
            raise ValueError(f"Unexpected magic number {magic} in label file {path}")
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return torch.tensor(labels, dtype=torch.long)

def load_fashion_mnist(root="./data/fashion-mnist"):
    paths = {k: _download_if_needed(v, root=root) for k, v in FASHION_MNIST_FILES.items()}
    x_train = _read_idx_images(paths["train_images"])
    y_train = _read_idx_labels(paths["train_labels"])
    x_test = _read_idx_images(paths["test_images"])
    y_test = _read_idx_labels(paths["test_labels"])
    return (x_train, y_train), (x_test, y_test)

# ---------------------------------------------------------------------
# Plotting utilities.
# ---------------------------------------------------------------------
def as_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def display_image(img):
    """Convert CHW or HWC image arrays/tensors into something matplotlib can show."""
    img = as_numpy(img)
    if img.ndim == 3 and img.shape[0] in (1, 3):
        img = np.moveaxis(img, 0, -1)
    if img.ndim == 3 and img.shape[-1] == 1:
        img = img[..., 0]
    return np.squeeze(img)

def plot_history(history, title="Training curve"):
    plt.figure(figsize=(6, 3.5))
    plt.plot(history["loss"], marker="o", label="train")
    if "val_loss" in history and len(history["val_loss"]) > 0:
        plt.plot(history["val_loss"], marker="o", label="validation")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

def show_label_grid(images, labels=None, n=12, title=None, cmap="gray"):
    images = as_numpy(images)
    n = min(n, len(images))
    cols = min(n, 6)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(1.8 * cols, 1.9 * rows))
    axes = np.array(axes).reshape(-1)
    for i, ax in enumerate(axes):
        ax.axis("off")
        if i >= n:
            continue
        ax.imshow(display_image(images[i]), cmap=cmap, vmin=0, vmax=1)
        if labels is not None:
            lab = labels[i]
            if isinstance(lab, torch.Tensor):
                lab = int(lab.item())
            if isinstance(lab, (int, np.integer)):
                lab = FASHION_LABELS[int(lab)]
            ax.set_title(str(lab), fontsize=9)
    if title:
        plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

def show_reconstruction_grid(originals, reconstructions, labels=None, n=10, title="Original vs reconstruction"):
    originals = as_numpy(originals)
    reconstructions = as_numpy(reconstructions)
    n = min(n, len(originals))
    fig, axes = plt.subplots(2, n, figsize=(1.6 * n, 3.4))
    for i in range(n):
        axes[0, i].imshow(display_image(originals[i]), cmap="gray", vmin=0, vmax=1)
        axes[0, i].axis("off")
        if labels is not None:
            lab = labels[i]
            if isinstance(lab, torch.Tensor):
                lab = int(lab.item())
            if isinstance(lab, (int, np.integer)):
                lab = FASHION_LABELS[int(lab)]
            axes[0, i].set_title(str(lab), fontsize=9)
        axes[1, i].imshow(display_image(reconstructions[i]), cmap="gray", vmin=0, vmax=1)
        axes[1, i].axis("off")
    axes[0, 0].set_ylabel("input", fontsize=11)
    axes[1, 0].set_ylabel("recon", fontsize=11)
    plt.suptitle(title, y=1.03)
    plt.tight_layout()
    plt.show()

def show_three_row_grid(clean, middle, reconstructed, labels=None, row_names=("clean", "noisy", "denoised"), n=10, title=None):
    clean = as_numpy(clean)
    middle = as_numpy(middle)
    reconstructed = as_numpy(reconstructed)
    n = min(n, len(clean))
    fig, axes = plt.subplots(3, n, figsize=(1.6 * n, 5.0))
    rows = [clean, middle, reconstructed]
    for r in range(3):
        for i in range(n):
            axes[r, i].imshow(display_image(rows[r][i]), cmap="gray", vmin=0, vmax=1)
            axes[r, i].axis("off")
            if r == 0 and labels is not None:
                lab = labels[i]
                if isinstance(lab, torch.Tensor):
                    lab = int(lab.item())
                if isinstance(lab, (int, np.integer)):
                    lab = FASHION_LABELS[int(lab)]
                axes[r, i].set_title(str(lab), fontsize=9)
        axes[r, 0].set_ylabel(row_names[r], fontsize=11)
    if title:
        plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

def show_error_maps(originals, reconstructions, errors=None, n=8, title="High-error examples"):
    originals = as_numpy(originals)
    reconstructions = as_numpy(reconstructions)
    n = min(n, len(originals))
    fig, axes = plt.subplots(3, n, figsize=(1.6 * n, 5.0))
    for i in range(n):
        orig = display_image(originals[i])
        rec = display_image(reconstructions[i])
        err_map = np.abs(orig - rec)

        axes[0, i].imshow(orig, cmap="gray", vmin=0, vmax=1)
        axes[1, i].imshow(rec, cmap="gray", vmin=0, vmax=1)
        axes[2, i].imshow(err_map, cmap="magma")
        for r in range(3):
            axes[r, i].axis("off")
        if errors is not None:
            axes[0, i].set_title(f"{errors[i]:.4f}", fontsize=9)

    axes[0, 0].set_ylabel("input", fontsize=11)
    axes[1, 0].set_ylabel("recon", fontsize=11)
    axes[2, 0].set_ylabel("|error|", fontsize=11)
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

def reconstruction_mse(x_true, x_pred):
    x_true = as_numpy(x_true)
    x_pred = as_numpy(x_pred)
    axes = tuple(range(1, x_true.ndim))
    return np.mean((x_true - x_pred) ** 2, axis=axes)

def plot_error_histogram(normal_errors, anomaly_errors=None, threshold=None, title="Reconstruction error"):
    plt.figure(figsize=(7, 4))
    plt.hist(normal_errors, bins=40, alpha=0.75, label="normal")
    if anomaly_errors is not None:
        plt.hist(anomaly_errors, bins=40, alpha=0.65, label="anomaly")
    if threshold is not None:
        plt.axvline(threshold, linestyle="--", linewidth=2, label=f"threshold = {threshold:.4f}")
    plt.xlabel("reconstruction MSE")
    plt.ylabel("number of images")
    plt.title(title)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()

def draw_autoencoder_diagram(latent_dim=64):
    from matplotlib.patches import Rectangle, FancyArrowPatch

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    boxes = [
        ("input\n1 x 28 x 28", 0.03, 0.34, 0.12, 0.32, "#F2F2F2"),
        ("encoder\ncompress", 0.23, 0.34, 0.16, 0.32, "#CDECCB"),
        (f"bottleneck\nz in R^{latent_dim}", 0.46, 0.34, 0.16, 0.32, "#F6D7A7"),
        ("decoder\nreconstruct", 0.69, 0.34, 0.16, 0.32, "#C8D8F2"),
        ("output\n1 x 28 x 28", 0.89, 0.34, 0.10, 0.32, "#F2F2F2"),
    ]

    for text, x, y, w, h, color in boxes:
        rect = Rectangle((x, y), w, h, facecolor=color, edgecolor="black", linewidth=1.2)
        ax.add_patch(rect)
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=12)

    arrow_pairs = [(0.15, 0.23), (0.39, 0.46), (0.62, 0.69), (0.85, 0.89)]
    for x0, x1 in arrow_pairs:
        ax.add_patch(FancyArrowPatch((x0, 0.50), (x1, 0.50), arrowstyle="->", mutation_scale=18, linewidth=1.5))

    ax.text(0.50, 0.12, "training target is the input itself: minimize reconstruction error", ha="center", fontsize=12)
    plt.show()

# ---------------------------------------------------------------------
# Models and training utilities.
# ---------------------------------------------------------------------
class DenseAutoencoder(nn.Module):
    def __init__(self, latent_dim=64, hidden_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_dim),
            nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim, 28 * 28),
            nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28)),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        return self.decode(self.encode(x))

class ConvDenoisingAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2),          # 28 x 28 -> 14 x 14
            nn.Conv2d(16, 8, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2),          # 14 x 14 -> 7 x 7
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(8, 8, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.Conv2d(16, 1, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        return self.decode(self.encode(x))

def build_dense_autoencoder(latent_dim=64, hidden_dim=128):
    return DenseAutoencoder(latent_dim=latent_dim, hidden_dim=hidden_dim)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def make_loader(x_inputs, x_targets, batch_size=BATCH_SIZE, shuffle=True):
    dataset = TensorDataset(x_inputs.float(), x_targets.float())
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

def train_autoencoder(model, train_loader, val_loader=None, epochs=5, lr=1e-3, device=DEVICE, verbose=True):
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"loss": [], "val_loss": []}

    for epoch in range(1, epochs + 1):
        model.train()
        train_total = 0.0
        train_count = 0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()

            batch_size = xb.size(0)
            train_total += loss.item() * batch_size
            train_count += batch_size

        train_loss = train_total / train_count
        history["loss"].append(train_loss)

        if val_loader is not None:
            model.eval()
            val_total = 0.0
            val_count = 0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb = xb.to(device)
                    yb = yb.to(device)
                    pred = model(xb)
                    loss = criterion(pred, yb)
                    batch_size = xb.size(0)
                    val_total += loss.item() * batch_size
                    val_count += batch_size
            val_loss = val_total / val_count
            history["val_loss"].append(val_loss)
            if verbose:
                print(f"Epoch {epoch:02d}/{epochs} | train loss {train_loss:.5f} | val loss {val_loss:.5f}")
        elif verbose:
            print(f"Epoch {epoch:02d}/{epochs} | train loss {train_loss:.5f}")

    return history

@torch.no_grad()
def reconstruct(model, x, batch_size=BATCH_SIZE, device=DEVICE):
    model.eval()
    model.to(device)
    x = x.float()
    outputs = []
    for start in range(0, len(x), batch_size):
        xb = x[start:start + batch_size].to(device)
        outputs.append(model(xb).cpu())
    return torch.cat(outputs, dim=0)

@torch.no_grad()
def encode_data(model, x, batch_size=BATCH_SIZE, device=DEVICE):
    model.eval()
    model.to(device)
    x = x.float()
    outputs = []
    for start in range(0, len(x), batch_size):
        xb = x[start:start + batch_size].to(device)
        outputs.append(model.encode(xb).cpu())
    return torch.cat(outputs, dim=0)


---
## Exercise 1: Basic reconstruction with a dense auto-encoder

An auto-encoder learns

$$
x \rightarrow z \rightarrow \hat{x}.
$$


The target is not a label. The target is the same image \(x\). The bottleneck \(z\) should keep enough information to reconstruct the image.

### Task
1. Load Fashion-MNIST and visualize the images.
2. Build a dense auto-encoder with a bottleneck.
3. Train it with mean squared reconstruction loss.
4. Compare original images with reconstructions.


In [ ]:
(x_train_full, y_train_full), (x_test_full, y_test_full) = load_fashion_mnist()

rng = torch.Generator().manual_seed(SEED)
train_idx = torch.randperm(len(x_train_full), generator=rng)[:TRAIN_SUBSET]
test_idx = torch.randperm(len(x_test_full), generator=rng)[:TEST_SUBSET]

x_train = x_train_full[train_idx]
y_train = y_train_full[train_idx]
x_test = x_test_full[test_idx]
y_test = y_test_full[test_idx]

train_loader = make_loader(x_train, x_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = make_loader(x_test, x_test, batch_size=BATCH_SIZE, shuffle=False)

print("x_train:", tuple(x_train.shape))
print("x_test: ", tuple(x_test.shape))

show_label_grid(x_train[:12], y_train[:12], n=12, title="Fashion-MNIST examples")
draw_autoencoder_diagram(latent_dim=64)


In [ ]:
# TODO: choose the size of the bottleneck.
latent_dim = ...

# TODO: build the dense auto-encoder.
autoencoder = ...

print(autoencoder)
print("Trainable parameters:", count_parameters(autoencoder))


In [ ]:
history = train_autoencoder(
    autoencoder,
    train_loader,
    val_loader=test_loader,
    epochs=20,
    lr=1e-3,
    device=DEVICE
)

plot_history(history, title="Dense auto-encoder reconstruction loss")


In [ ]:
sample = x_test[:12]
sample_labels = y_test[:12]

recon = reconstruct(autoencoder, sample, device=DEVICE)
show_reconstruction_grid(sample, recon, labels=sample_labels, n=12, title="Dense AE: original vs reconstruction")

test_recon = reconstruct(autoencoder, x_test, device=DEVICE)
test_errors = reconstruction_mse(x_test, test_recon)

plt.figure(figsize=(7, 4))
plt.hist(test_errors, bins=50)
plt.xlabel("reconstruction MSE")
plt.ylabel("number of test images")
plt.title("Distribution of reconstruction error on test images")
plt.grid(alpha=0.25)
plt.show()

print(f"Mean test reconstruction MSE: {test_errors.mean():.5f}")


### Things to try
- Change `latent_dim` to `8`, `32`, `128`. What changes in the images?
- Look at the histogram. Are all classes reconstructed equally well?
- Increase the hidden layer width from `128` to `256`.


---
## Exercise 2: The bottleneck as visual compression

The bottleneck controls how many numbers the model can use to describe an image. A tiny bottleneck should reconstruct only the most global structure. A larger bottleneck can keep more details.

### Task
1. Train the same architecture with several bottleneck sizes.
2. Show reconstructions of the same images for each bottleneck size.
3. Use a 2D bottleneck to draw a latent-space map colored by clothing class.


In [ ]:
# This cell trains several small models. It is intentionally visual.
# TODO: choose at least three bottleneck sizes.
# Make sure 2 is included, because the next cell draws a 2D latent map.
latent_dims = [ ... ]
sweep_epochs = ...

sweep_models = {}
probe = x_test[:8]
probe_labels = y_test[:8]

for d in latent_dims:
    ae_d = build_dense_autoencoder(latent_dim=d, hidden_dim=128).to(DEVICE)
    print(f"\nTraining bottleneck size d = {d}")
    train_autoencoder(
        ae_d,
        train_loader,
        val_loader=test_loader,
        epochs=sweep_epochs,
        lr=1e-3,
        device=DEVICE,
        verbose=False
    )
    sweep_models[d] = (ae_d, reconstruct(ae_d, probe, device=DEVICE))

print("Done.")


In [ ]:
rows = 1 + len(latent_dims)
cols = len(probe)
fig, axes = plt.subplots(rows, cols, figsize=(1.55 * cols, 1.75 * rows))

for j in range(cols):
    axes[0, j].imshow(display_image(probe[j]), cmap="gray", vmin=0, vmax=1)
    axes[0, j].axis("off")
    axes[0, j].set_title(FASHION_LABELS[int(probe_labels[j])], fontsize=8)
axes[0, 0].set_ylabel("input", fontsize=10)

for r, d in enumerate(latent_dims, start=1):
    recon_d = sweep_models[d][1]
    for j in range(cols):
        axes[r, j].imshow(display_image(recon_d[j]), cmap="gray", vmin=0, vmax=1)
        axes[r, j].axis("off")
    axes[r, 0].set_ylabel(f"z={d}", fontsize=10)

plt.suptitle("Same images reconstructed with different bottleneck sizes", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Use the 2D encoder from the sweep to make a latent-space scatter plot.
# TODO: make sure your `latent_dims` list above contains 2.
ae2, _ = ...

z2 = encode_data(ae2, x_test, device=DEVICE).numpy()

plt.figure(figsize=(7, 6))
scatter = plt.scatter(z2[:, 0], z2[:, 1], c=y_test.numpy(), cmap="tab10", s=8, alpha=0.75)
cbar = plt.colorbar(scatter, ticks=range(10))
cbar.ax.set_yticklabels(FASHION_LABELS)
plt.xlabel("latent coordinate 1")
plt.ylabel("latent coordinate 2")
plt.title("2D latent space learned by an auto-encoder")
plt.grid(alpha=0.2)
plt.show()

# now do T-SNE visualisation





### Questions
- Which classes cluster together in the 2D map?
- Why does a 2D auto-encoder map not separate all classes perfectly?
- What gets lost first when the bottleneck is very small: outline, texture, or fine details?


---
## Exercise 3: De-noising auto-encoder

A de-noising auto-encoder receives a corrupted image \(\tilde{x}\) but is trained to reconstruct the clean image \(x\):

$$
\tilde{x} \rightarrow z \rightarrow \hat{x} \approx x.
$$

This forces the code to capture stable patterns instead of copying every noisy pixel.

### Task
1. Add Gaussian noise to the images.
2. Build a small convolutional auto-encoder.
3. Train on noisy inputs and clean targets.
4. Visualize clean, noisy, and de-noised images side by side.


In [ ]:
# TODO: choose a noise factor. Try 0.35 first.
noise_factor = ...

noise_gen = torch.Generator().manual_seed(123)
x_train_noisy = x_train + noise_factor * torch.randn(x_train.shape, generator=noise_gen)
x_test_noisy = x_test + noise_factor * torch.randn(x_test.shape, generator=noise_gen)

x_train_noisy = torch.clamp(x_train_noisy, 0.0, 1.0)
x_test_noisy = torch.clamp(x_test_noisy, 0.0, 1.0)

show_three_row_grid(
    x_test[:10], x_test_noisy[:10], x_test_noisy[:10],
    labels=y_test[:10],
    row_names=("clean", "noisy", "noisy again"),
    title="Noise corruption before training"
)


In [ ]:
# TODO: instantiate the convolutional de-noising auto-encoder.
denoiser = ...

print(denoiser)
print("Trainable parameters:", count_parameters(denoiser))


In [ ]:
denoise_train_loader = make_loader(x_train_noisy, x_train, batch_size=BATCH_SIZE, shuffle=True)
denoise_test_loader = make_loader(x_test_noisy, x_test, batch_size=BATCH_SIZE, shuffle=False)

history_denoise = train_autoencoder(
    denoiser,
    denoise_train_loader,
    val_loader=denoise_test_loader,
    epochs=6,
    lr=1e-3,
    device=DEVICE
)

plot_history(history_denoise, title="Denoising auto-encoder loss")


In [ ]:
denoised = reconstruct(denoiser, x_test_noisy[:12], device=DEVICE)

show_three_row_grid(
    x_test[:12], x_test_noisy[:12], denoised,
    labels=y_test[:12],
    row_names=("clean target", "noisy input", "denoised"),
    title="De-noising AE: clean vs noisy vs output"
)

show_error_maps(
    x_test[:8],
    denoised[:8],
    errors=reconstruction_mse(x_test[:8], denoised[:8]),
    n=8,
    title="Where does the de-noising model still make mistakes?"
)


---
## Exercise 4: Look inside the convolutional bottleneck

A convolutional auto-encoder does not have to compress the whole image into one vector. Here the bottleneck is a stack of small feature maps. Each channel can respond to different visual patterns.

### Task
1. Extract the output of the convolutional encoder.
2. Visualize the feature maps for one noisy image.
3. Compare the feature maps with the input image.


In [ ]:
# TODO


### Things to try
- Change the selected `idx` and see whether the feature maps respond to different parts of the image.
- Change `noise_factor` and compare how the bottleneck activations change.
- Replace `MaxPool2d` with strided convolutions and compare the de-noising results.


---
## Exercise 5: Anomaly detection with reconstruction error

This exercise has only two parts.

The key idea is simple: train the auto-encoder only on **normal** data. At test time, examples from a different distribution should reconstruct badly. Large reconstruction error means "possibly anomalous."

### Task
**(a)** Train an auto-encoder on one normal class only.  
**(b)** Use reconstruction MSE as an anomaly score and visualize the threshold.


In [ ]:
# Part (a): train only on one normal class.
# Fashion-MNIST label 1 is Trouser. Label 9 is Ankle boot.
normal_class = 1
anomaly_class = 9

normal_train = x_train_full[y_train_full == normal_class][:4000]
normal_test = x_test_full[y_test_full == normal_class][:1000]
anomaly_test = x_test_full[y_test_full == anomaly_class][:1000]

print("Normal train:", tuple(normal_train.shape), FASHION_LABELS[normal_class])
print("Normal test: ", tuple(normal_test.shape))
print("Anomaly test:", tuple(anomaly_test.shape), FASHION_LABELS[anomaly_class])

show_label_grid(normal_train[:8], [normal_class] * 8, n=8, title="Training data: normal class only")
show_label_grid(anomaly_test[:8], [anomaly_class] * 8, n=8, title="Held-out anomaly class")

# TODO: build a dense auto-encoder for normal-only training.
anom_ae = ...

normal_train_loader = make_loader(normal_train, normal_train, batch_size=BATCH_SIZE, shuffle=True)
normal_test_loader = make_loader(normal_test, normal_test, batch_size=BATCH_SIZE, shuffle=False)

history_anom = train_autoencoder(
    anom_ae,
    normal_train_loader,
    val_loader=normal_test_loader,
    epochs=8,
    lr=1e-3,
    device=DEVICE
)

plot_history(history_anom, title="Normal-only auto-encoder loss")


In [ ]:
# Part (b): reconstruction error as anomaly score.
normal_train_recon = reconstruct(anom_ae, normal_train, device=DEVICE)
normal_train_error = reconstruction_mse(normal_train, normal_train_recon)

# Simple threshold: one standard deviation above the mean normal training error.
threshold = normal_train_error.mean() + normal_train_error.std()

x_eval = torch.cat([normal_test, anomaly_test], dim=0)
y_eval = np.array([0] * len(normal_test) + [1] * len(anomaly_test))  # 0 = normal, 1 = anomaly

eval_recon = reconstruct(anom_ae, x_eval, device=DEVICE)
eval_error = reconstruction_mse(x_eval, eval_recon)
pred_anomaly = (eval_error > threshold).astype(int)

normal_errors = eval_error[y_eval == 0]
anomaly_errors = eval_error[y_eval == 1]

plot_error_histogram(
    normal_errors,
    anomaly_errors,
    threshold=threshold,
    title="Anomaly detection by reconstruction error"
)

print("Confusion matrix with rows=true and columns=predicted")
print(confusion_matrix(y_eval, pred_anomaly))
print()
print(classification_report(y_eval, pred_anomaly, target_names=["normal", "anomaly"]))

# Visualize the easiest and hardest examples for the model.
order = np.argsort(eval_error)
low = order[:8]
high = order[-8:][::-1].copy()

show_reconstruction_grid(
    x_eval[low], eval_recon[low],
    labels=["normal" if y_eval[i] == 0 else "anomaly" for i in low],
    n=8,
    title="Lowest reconstruction errors"
)

show_error_maps(
    x_eval[high], eval_recon[high],
    errors=eval_error[high],
    n=8,
    title="Highest reconstruction errors and error maps"
)


### Questions
- Is the threshold separating the two error distributions cleanly?
- Which normal examples have high reconstruction error? What makes them unusual within their class?
- Which anomaly examples have low reconstruction error? What makes them similar to the normal class?
